# Project 7: Mutational Signatures in African Ancestry Cancer Patients

**Author:** Karimat Adeola Busari  
**Date:** July 2026  
**Portfolio:** Computational Cancer Genomics — African and Underserved Populations

---

## Biological Question
What mutational processes are driving cancer in patients of African ancestry, and how do they compare to COSMIC reference signatures derived predominantly from Western cohorts?

## How to Use This Template
This notebook is designed as a reusable pipeline. To run it on your own data:
1. Update the file paths in the **CONFIG** cell below
2. Update the `POPULATION_FILTER` variable to match your population of interest
3. Run all cells in order

**Input required:** A MAF file with standard TCGA columns, a clinical patient file with demographic data, and a clinical sample file linking patients to samples.

---

## Pipeline Overview
1. Setup & Installation
2. Data Loading (CONFIG)
3. Data Filtering by Population
4. Trinucleotide Context Extraction
5. 96-Context Mutation Matrix Construction
6. NMF Signature Extraction
7. COSMIC Signature Matching
8. Visualisation & Output

---
## Section 1: Setup & Installation
Install required packages. Re-run this cell at the start of every new Colab session.

In [ ]:
!pip install synapseclient pyfaidx tqdm --quiet
print('All packages installed successfully!')

---
## Section 2: Mount Google Drive
Mount Google Drive to access saved data files. Required at the start of every session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

---
## Section 3: CONFIG — Update These Paths for Your Data

**This is the only section you need to change when using a new dataset.**

- `MAF_PATH`: Path to your somatic mutation MAF file
- `CLINICAL_PATIENT_PATH`: Path to clinical patient file (must contain race/ancestry column)
- `CLINICAL_SAMPLE_PATH`: Path to clinical sample file (links patients to samples)
- `REFERENCE_GENOME_PATH`: Path to GRCh37 or GRCh38 FASTA reference genome
- `COSMIC_PATH`: Path to COSMIC SBS signatures file (download from cancer.sanger.ac.uk/signatures)
- `OUTPUT_DIR`: Where to save results
- `POPULATION_FILTER`: The ancestry/race value to filter on
- `RACE_COLUMN`: The column name in your clinical file that contains race/ancestry information
- `N_SIGNATURES`: Number of NMF signatures to extract (use elbow plot to determine)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================
# CONFIG — CHANGE ONLY THESE PATHS FOR A NEW DATASET
# ============================================================

MAF_PATH = '/content/drive/MyDrive/Project7/data_mutations_extended.txt'
CLINICAL_PATIENT_PATH = '/content/drive/MyDrive/Project7/data_clinical_patient.txt'
CLINICAL_SAMPLE_PATH = '/content/drive/MyDrive/Project7/data_clinical_sample.txt'
REFERENCE_GENOME_PATH = '/content/drive/MyDrive/Project7/reference_genome/GRCh37.fa'
COSMIC_PATH = '/content/drive/MyDrive/Project7/COSMIC_v3.3.1_SBS_GRCh37.txt'
OUTPUT_DIR = '/content/drive/MyDrive/Project7/'

POPULATION_FILTER = 'Black'   # Change to your population of interest
RACE_COLUMN = 'PRIMARY_RACE'  # Change to match your clinical file's column name
N_SIGNATURES = 5              # Change after reviewing elbow plot

# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('CONFIG loaded successfully!')
print(f'Population filter: {POPULATION_FILTER}')
print(f'Output directory: {OUTPUT_DIR}')

---
## Section 4: Load Data
Load the three input files into pandas dataframes. This may take 1-2 minutes for large MAF files.

In [ ]:
print('Loading data files...')

maf = pd.read_csv(MAF_PATH, sep='\t', comment='#', low_memory=False)
clinical_patient = pd.read_csv(CLINICAL_PATIENT_PATH, sep='\t', comment='#', low_memory=False)
clinical_sample = pd.read_csv(CLINICAL_SAMPLE_PATH, sep='\t', comment='#', low_memory=False)

print(f'MAF file: {maf.shape[0]:,} mutations x {maf.shape[1]} columns')
print(f'Clinical patient: {clinical_patient.shape[0]:,} patients x {clinical_patient.shape[1]} columns')
print(f'Clinical sample: {clinical_sample.shape[0]:,} samples x {clinical_sample.shape[1]} columns')

---
## Section 5: Filter by Population

Merge clinical files and filter for the target population.

**Strategy:**
1. Merge clinical_patient + clinical_sample on PATIENT_ID (adds race/ancestry info to each sample)
2. Filter by POPULATION_FILTER (defined in CONFIG)
3. Merge with MAF on SAMPLE_ID = Tumor_Sample_Barcode
4. Keep only Single Base Substitutions (SNP) for SBS signature analysis

**Note on population labelling:** GENIE uses self-reported race categories. 'Black' in GENIE refers to Black/African American patients treated at Western cancer centres. This is NOT equivalent to continental African patients. When true continental African MAF data becomes available, swap the MAF_PATH in CONFIG and update POPULATION_FILTER accordingly.

In [ ]:
# Step 1: Merge clinical patient + clinical sample
clinical_combined = pd.merge(clinical_patient, clinical_sample, on='PATIENT_ID')
print(f'Combined clinical file: {clinical_combined.shape}')

# Step 2: Filter for target population
filtered_clinical = clinical_combined[clinical_combined[RACE_COLUMN] == POPULATION_FILTER].copy()
print(f'After population filter ({POPULATION_FILTER}): {filtered_clinical.shape[0]:,} samples')
print(f'Unique patients: {filtered_clinical["PATIENT_ID"].nunique():,}')

# Step 3: Merge with MAF file
maf_filtered = pd.merge(
    filtered_clinical, maf,
    left_on='SAMPLE_ID',
    right_on='Tumor_Sample_Barcode'
)
print(f'After MAF merge: {maf_filtered.shape[0]:,} mutations')

# Step 4: Keep only SNPs for SBS analysis
maf_filtered = maf_filtered[maf_filtered['Variant_Type'] == 'SNP'].copy()
print(f'After SNP filter: {maf_filtered.shape[0]:,} single base substitutions')
print(f'Unique patients retained: {maf_filtered["PATIENT_ID"].nunique():,}')

---
## Section 6: Trinucleotide Context Extraction

For each mutation, extract the trinucleotide context (the mutated base plus its immediate left and right neighbours) from the reference genome using pyfaidx.

All contexts are normalised to the pyrimidine strand convention (C or T as the mutated base), reducing 192 possible contexts to the standard 96.

In [ ]:
from pyfaidx import Fasta
from tqdm import tqdm
tqdm.pandas()

# Load reference genome (indexed automatically on first run)
print('Loading reference genome...')
genome = Fasta(REFERENCE_GENOME_PATH)
print(f'Genome loaded. Sample chromosomes: {list(genome.keys())[:5]}')

In [ ]:
def get_trinucleotide(chrom, pos, genome):
    """
    Retrieve the trinucleotide context for a mutation.
    
    Parameters:
        chrom: chromosome (e.g. '1', 'X', 'MT')
        pos: 1-based genomic position of the mutation
        genome: pyfaidx Fasta object
    
    Returns:
        3-character string (left_base + ref_base + right_base), uppercase
    """
    try:
        # Handle chromosome naming: add 'chr' prefix, MT -> chrM
        if str(chrom) == 'MT':
            chrom_key = 'chrM'
        else:
            chrom_key = 'chr' + str(chrom)
        context = genome[chrom_key][pos-2:pos+1]
        return str(context).upper()
    except Exception:
        return None


def normalize_to_pyrimidine(category, ref):
    """
    Normalise mutation context to pyrimidine strand convention.
    If reference allele is G or A, take reverse complement.
    
    Parameters:
        category: mutation category string e.g. 'G[G>A]A'
        ref: reference allele (single base)
    
    Returns:
        Normalised category string with C or T as the mutated base
    """
    if ref in ['C', 'T']:
        return category
    comp = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    left = comp[category[0]]
    ref_comp = comp[category[2]]
    alt_comp = comp[category[4]]
    right = comp[category[6]]
    return f"{right}[{ref_comp}>{alt_comp}]{left}"


print('Functions defined successfully!')

In [ ]:
# Extract trinucleotide context for all mutations
# Note: progress_apply shows a progress bar — this runs quickly (~10-15 seconds)
print('Extracting trinucleotide contexts...')
maf_filtered['trinucleotide'] = maf_filtered.progress_apply(
    lambda row: get_trinucleotide(row['Chromosome'], row['Start_Position'], genome),
    axis=1
)

# Drop rows where context could not be extracted (e.g. edge of chromosome)
before = len(maf_filtered)
maf_filtered = maf_filtered.dropna(subset=['trinucleotide'])
after = len(maf_filtered)
print(f'Dropped {before - after} mutations with missing context (edge cases)')
print(f'Mutations retained: {after:,}')

In [ ]:
# Build mutation category labels and normalise to pyrimidine strand
maf_filtered['mutation_category'] = (
    maf_filtered['trinucleotide'].str[0] +
    '[' + maf_filtered['Reference_Allele'] +
    '>' + maf_filtered['Tumor_Seq_Allele2'] + ']' +
    maf_filtered['trinucleotide'].str[2]
)

maf_filtered['mutation_category_norm'] = maf_filtered.apply(
    lambda row: normalize_to_pyrimidine(row['mutation_category'], row['Reference_Allele']),
    axis=1
)

unique_cats = maf_filtered['mutation_category_norm'].nunique()
print(f'Unique normalised mutation categories: {unique_cats} (expected: 96)')
print(maf_filtered['mutation_category_norm'].head(5))

---
## Section 7: Build the 96-Context Mutation Matrix

Construct a patient × 96-context count matrix where each cell represents the number of mutations of a given trinucleotide context type observed in a given patient's tumour. This is the input to NMF.

In [ ]:
# Build patient x 96-context mutation count matrix
mutation_matrix = maf_filtered.groupby(
    ['PATIENT_ID', 'mutation_category_norm']
).size().unstack(fill_value=0)

print(f'Mutation matrix shape: {mutation_matrix.shape}')
print(f'Expected: ~{maf_filtered["PATIENT_ID"].nunique()} patients x 96 contexts')

# Save matrix
mutation_matrix.to_csv(OUTPUT_DIR + 'mutation_matrix_96.csv')
print(f'Matrix saved to {OUTPUT_DIR}mutation_matrix_96.csv')

---
## Section 8: NMF Signature Extraction

Apply Non-negative Matrix Factorization (NMF) to decompose the mutation matrix into:
- **Signatures matrix** (k × 96): the profile of each extracted signature
- **Exposures matrix** (patients × k): the contribution of each signature to each patient

First, determine the optimal number of signatures using the elbow method.

In [ ]:
from sklearn.decomposition import NMF

# Test k = 2 to 8 and record reconstruction error
print('Testing different numbers of signatures...')
reconstruction_errors = {}

for n in range(2, 9):
    model = NMF(n_components=n, random_state=42, max_iter=1000)
    model.fit_transform(mutation_matrix)
    reconstruction_errors[n] = model.reconstruction_err_
    print(f'  k={n}: reconstruction error = {model.reconstruction_err_:.2f}')

print('Done!')

In [ ]:
# Plot elbow curve to determine optimal number of signatures
plt.figure(figsize=(8, 4))
plt.plot(
    list(reconstruction_errors.keys()),
    list(reconstruction_errors.values()),
    marker='o', color='steelblue', linewidth=2
)
plt.xlabel('Number of Signatures (k)', fontsize=12)
plt.ylabel('Reconstruction Error', fontsize=12)
plt.title('NMF Reconstruction Error vs Number of Signatures\n(Choose k at the elbow — where the curve flattens)', fontsize=12)
plt.xticks(range(2, 9))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'reconstruction_error.png', dpi=150)
plt.show()
print(f'Elbow plot saved to {OUTPUT_DIR}reconstruction_error.png')
print(f'\nUpdate N_SIGNATURES in CONFIG to your chosen k value, then run the next cell.')

In [ ]:
# Extract final signatures using optimal k (set N_SIGNATURES in CONFIG)
print(f'Extracting {N_SIGNATURES} signatures...')

model_final = NMF(n_components=N_SIGNATURES, random_state=42, max_iter=1000)
W = model_final.fit_transform(mutation_matrix)  # patients x signatures
H = model_final.components_                     # signatures x 96 contexts

sig_labels = [f'Signature_{i+1}' for i in range(N_SIGNATURES)]

signatures_df = pd.DataFrame(H, columns=mutation_matrix.columns, index=sig_labels)
exposures_df = pd.DataFrame(W, columns=sig_labels, index=mutation_matrix.index)

print(f'Signatures matrix: {signatures_df.shape} (signatures x 96 contexts)')
print(f'Exposures matrix: {exposures_df.shape} (patients x signatures)')

signatures_df.to_csv(OUTPUT_DIR + 'signatures.csv')
exposures_df.to_csv(OUTPUT_DIR + 'exposures.csv')
print('Signatures and exposures saved!')

---
## Section 9: Visualise Extracted Signatures

Plot each extracted signature as a bar chart across the 96 trinucleotide contexts.
Colours represent the 6 mutation types: C>A (blue), C>G (black), C>T (red), T>A (grey), T>C (green), T>G (pink).

In [ ]:
# Colour map for the 6 SBS mutation types
MUTATION_COLORS = {
    'C>A': 'steelblue',
    'C>G': 'black',
    'C>T': 'red',
    'T>A': 'grey',
    'T>C': 'green',
    'T>G': 'pink'
}

def get_color(category):
    mut = category[2:5]  # Extract e.g. 'C>T' from 'A[C>T]G'
    return MUTATION_COLORS.get(mut, 'purple')

fig, axes = plt.subplots(N_SIGNATURES, 1, figsize=(20, 4 * N_SIGNATURES))

if N_SIGNATURES == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    sig = signatures_df.iloc[i]
    colors = [get_color(cat) for cat in sig.index]
    ax.bar(range(96), sig.values, color=colors, width=0.8)
    ax.set_title(f'Signature {i+1}', fontsize=13, fontweight='bold')
    ax.set_xticks([])
    ax.set_ylabel('Mutation probability', fontsize=10)
    ax.set_xlim(-1, 96)

plt.suptitle(
    f'Extracted Mutational Signatures — {POPULATION_FILTER} Cancer Patients (GENIE)',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'signatures_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Signatures plot saved to {OUTPUT_DIR}signatures_plot.png')

---
## Section 10: COSMIC Signature Matching

Compare each extracted signature against all 79 COSMIC v3.3.1 SBS reference signatures using cosine similarity.

**Interpretation guide:**
- Cosine similarity > 0.80: High confidence match
- Cosine similarity 0.70–0.80: Moderate confidence
- Cosine similarity < 0.70: Interpret with caution — may represent a mixed or novel signature

**Download COSMIC signatures from:** https://cancer.sanger.ac.uk/signatures/downloads/

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Load COSMIC reference signatures
cosmic = pd.read_csv(COSMIC_PATH, sep='\t', index_col=0)
print(f'COSMIC signatures loaded: {cosmic.shape} (96 contexts x {cosmic.shape[1]} signatures)')

In [ ]:
# Align contexts between extracted signatures and COSMIC
cosmic_aligned = cosmic.loc[signatures_df.columns]

# Calculate cosine similarity
similarity_matrix = cosine_similarity(signatures_df.values, cosmic_aligned.T.values)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=sig_labels,
    columns=cosmic_aligned.columns
)

# Print best matches
print('=== COSMIC Signature Matching Results ===')
print()
results = []
for sig in similarity_df.index:
    best_match = similarity_df.loc[sig].idxmax()
    best_score = similarity_df.loc[sig].max()
    confidence = 'HIGH' if best_score >= 0.80 else 'MODERATE' if best_score >= 0.70 else 'LOW'
    print(f'{sig} → {best_match} (cosine similarity: {best_score:.3f}) [{confidence} confidence]')
    results.append({'Signature': sig, 'Best_COSMIC_Match': best_match,
                    'Cosine_Similarity': round(best_score, 3), 'Confidence': confidence})

results_df = pd.DataFrame(results)

# Save results
similarity_df.to_csv(OUTPUT_DIR + 'cosmic_similarity_full.csv')
results_df.to_csv(OUTPUT_DIR + 'cosmic_best_matches.csv', index=False)
print(f'\nResults saved to {OUTPUT_DIR}')

---
## Section 11: Summary of Outputs

All outputs are saved to the OUTPUT_DIR defined in CONFIG.

In [ ]:
print('=== PROJECT 7 PIPELINE COMPLETE ===')
print()
print('Files saved to:', OUTPUT_DIR)
print()
print('  mutation_matrix_96.csv     — Patient x 96-context mutation count matrix')
print('  signatures.csv             — Extracted NMF signatures (k x 96 contexts)')
print('  exposures.csv              — Patient signature exposures (patients x k)')
print('  cosmic_similarity_full.csv — Full cosine similarity matrix vs all COSMIC signatures')
print('  cosmic_best_matches.csv    — Best COSMIC match per extracted signature')
print('  reconstruction_error.png   — NMF elbow plot for signature number selection')
print('  signatures_plot.png        — Bar chart visualisation of extracted signatures')
print()
print('Key findings:')
for _, row in results_df.iterrows():
    print(f"  {row['Signature']} → {row['Best_COSMIC_Match']} ({row['Cosine_Similarity']}) [{row['Confidence']}]")

---
## Notes for Reproducibility

**Data source:** AACR Project GENIE v19.0 (synapse.org, syn7222066)  
**Reference genome:** GRCh37/hg19 (UCSC hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/)  
**COSMIC signatures:** v3.3.1 SBS GRCh37 (cancer.sanger.ac.uk/signatures/downloads/)  
**Python version:** 3.x  
**Key libraries:** pandas, numpy, scikit-learn, pyfaidx, matplotlib, tqdm

**To reproduce on a new dataset:**  
1. Update all paths in the CONFIG cell (Section 3)  
2. Update POPULATION_FILTER and RACE_COLUMN to match your data  
3. Run all cells in order  
4. Review the elbow plot and update N_SIGNATURES if needed  

**Limitation note:** The COSMIC reference signatures were derived predominantly from European and Asian cancer cohorts. Matching scores below 0.80 may indicate population-specific mutational processes not yet captured in the COSMIC catalogue — particularly relevant for continental African patient data.